# 07 · Especialista prestart H60 — candidato congelado y lectura final

**Qué pregunta responde.** ¿Cuál es el candidato operativo final del arco y qué soporte real tiene? Es el puente entre la escalera de modelos y el resultado final de la memoria: el candidato no es la red más compleja, sino el especialista que mejor sobrevive al protocolo económico.

**Qué entradas usa.** Solo CSV públicos: `results/key_results.csv`, `final_candidate_summary.csv` (resumen del bloque fresh) y el ledger anonimizado `final_candidate_actions_anonymized.csv`. Sin corpus privado y sin aleatoriedad: cuaderno de lectura re-ejecutable.

**Qué produce.** La regla base congelada (especialista `no_clock`, universo `strict_45_60_early` de −60 a −45 s, H60, EV > 0,75, health ≥ 0,50), el resultado OOS limpio sin filtro de volatilidad, el diagnóstico post-hoc por régimen de volatilidad y el desglose diario del ledger.

**Conclusión (2 frases).** El resultado limpio es una señal parcial fuera de muestra (+0,349 ticks netos @0,5 sobre 754 acciones en el bloque fresh, 3 de 5 días positivos); el filtro de baja volatilidad (+1,069 sobre 318 acciones) es una hipótesis post-hoc prometedora, no una segunda validación. La siguiente prueba válida exige fechas posteriores al 10-jun-2026 sin retocar umbrales — exactamente lo que quedó pre-registrado.

**Filas de `results/key_results.csv`.** `strategy,complex_v1a_prestart_h60_*` (resultado fresh sin gate y gate post-hoc), `oof,clean_h60_upstream,*` (la señal parcial upstream), `horizon,h60_paired`/`horizon,h120` (por qué H60 y no H120) y `complex_v1a_prestart_h60_paper_shadow_v1,*` (replay del bloque).

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'results').exists():
    ROOT = ROOT.parent

key = pd.read_csv(ROOT / 'results' / 'key_results.csv')
summary = pd.read_csv(ROOT / 'results' / 'final_candidate_summary.csv')
ledger = pd.read_csv(ROOT / 'results' / 'final_candidate_actions_anonymized.csv')

## Regla base congelada

Antes del bloque fresh estaban congelados el especialista `no_clock`, el universo `strict_45_60_early` y los umbrales de EV/health. El filtro de volatilidad se adopta después como diagnóstico, no como resultado confirmatorio.

In [2]:
pd.DataFrame([
    {"componente": "Universo", "valor": "strict_45_60_early (-60 a -45 s prestart)"},
    {"componente": "Horizonte", "valor": "H60"},
    {"componente": "EV", "valor": "> 0.75"},
    {"componente": "Health probability", "valor": ">= 0.50"},
    {"componente": "Volatilidad", "valor": "post-hoc: perp_realized_vol_bps_5s <= 0.6657"},
])

,componente,valor
0,Universo,strict_45_60_early (-60 a -45 s prestart)
1,Horizonte,H60
2,EV,> 0.75
3,Health probability,>= 0.50
4,Volatilidad,post-hoc: perp_realized_vol_bps_5s <= 0.6657


## Resultado OOS limpio

Esta es la cifra confirmatoria principal: sin filtro de volatilidad decidido tras mirar el bloque fresh.

In [3]:
base = summary[summary["scope"].eq("base_oos")].pivot_table(index="scope", columns="metric", values="value", aggfunc="first")
base[["n_actions", "mean_net_cost_0p25", "mean_net_cost_0p5", "mean_net_cost_1p0", "positive_days_cost_0p5", "n_days"]]

metric,n_actions,mean_net_cost_0p25,mean_net_cost_0p5,mean_net_cost_1p0,positive_days_cost_0p5,n_days
scope,,,,,,
base_oos,754.0,0.536987,0.349173,-0.026456,3.0,5.0


## Diagnóstico post-hoc de volatilidad

La baja volatilidad concentra la señal, pero esta separación se estudió después de inspeccionar el bloque. Por eso queda como hipótesis congelada pendiente de validación prospectiva.

In [4]:
regime = summary[summary["scope"].isin(["low_vol_posthoc", "high_vol_posthoc", "missing_volatility"])]
regime.pivot_table(index="scope", columns="metric", values="value", aggfunc="first")[["n_actions", "mean_net_cost_0p25", "mean_net_cost_0p5", "mean_net_cost_1p0"]]

metric,n_actions,mean_net_cost_0p25,mean_net_cost_0p5,mean_net_cost_1p0
scope,,,,
high_vol_posthoc,435.0,0.003656,-0.183493,-0.557791
low_vol_posthoc,318.0,1.257770,1.069000,0.691460
missing_volatility,1.0,3.326763,3.153524,2.807045


## Desglose diario del ledger público

El ledger anonimizado permite auditar directamente el soporte temporal de la memoria.

In [5]:
daily = (
    ledger.assign(low_vol=ledger["volatility_regime"].eq("low"))
    .groupby("session_day")
    .agg(
        base_actions=("action_id", "size"),
        base_net_0p5=("net_cost_0p5", "mean"),
        low_actions=("low_vol", "sum"),
    )
)
low_daily = ledger[ledger["volatility_regime"].eq("low")].groupby("session_day")["net_cost_0p5"].mean()
daily["low_net_0p5"] = low_daily
daily

,base_actions,base_net_0p5,low_actions,low_net_0p5
session_day,,,,
2026-06-06,220,0.666289,106,0.629230
2026-06-07,97,1.341818,42,3.571630
2026-06-08,241,0.307938,76,1.384584
2026-06-09,146,-0.564707,82,0.025325
2026-06-10,50,-0.104596,12,1.327516


## Decisión

El resultado limpio es una señal parcial fuera de muestra; el filtro de volatilidad es una hipótesis prometedora, no una segunda validación. La siguiente prueba válida requiere fechas posteriores al 10 de junio de 2026 sin retocar umbrales.